# Reproducing SILVA and Source Methods

This lab makes the reproduction boundary executable. It inspects all canonical
families, resolves aliases to their real constructor signatures, builds a
custom conditioned equilibrium, adapts a joint diffusion trajectory to an
observation-conditioned restoration step, and emits a structured run record.

The universal equation is

$$
z_0=I_\eta(x),\qquad z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

A source method is reproduced only when its equation, data release,
preprocessing, dimensions, numerical settings, training schedule, checkpoints,
seeds, and metrics are all declared. Compact checks verify mechanisms; they do
not stand in for an unexecuted published benchmark.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [1](https://jseluis.github.io/silva-networks/paper/references/#ref-1), [4](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [5](https://jseluis.github.io/silva-networks/paper/references/#ref-5), [7](https://jseluis.github.io/silva-networks/paper/references/#ref-7), [8](https://jseluis.github.io/silva-networks/paper/references/#ref-8), [9](https://jseluis.github.io/silva-networks/paper/references/#ref-9), [22](https://jseluis.github.io/silva-networks/paper/references/#ref-22), [23](https://jseluis.github.io/silva-networks/paper/references/#ref-23), [31](https://jseluis.github.io/silva-networks/paper/references/#ref-31), [32](https://jseluis.github.io/silva-networks/paper/references/#ref-32), [36](https://jseluis.github.io/silva-networks/paper/references/#ref-36), [37](https://jseluis.github.io/silva-networks/paper/references/#ref-37), [38](https://jseluis.github.io/silva-networks/paper/references/#ref-38), [43](https://jseluis.github.io/silva-networks/paper/references/#ref-43), [44](https://jseluis.github.io/silva-networks/paper/references/#ref-44), [45](https://jseluis.github.io/silva-networks/paper/references/#ref-45), [46](https://jseluis.github.io/silva-networks/paper/references/#ref-46), [47](https://jseluis.github.io/silva-networks/paper/references/#ref-47), [48](https://jseluis.github.io/silva-networks/paper/references/#ref-48), [49](https://jseluis.github.io/silva-networks/paper/references/#ref-49), [50](https://jseluis.github.io/silva-networks/paper/references/#ref-50), [51](https://jseluis.github.io/silva-networks/paper/references/#ref-51), [52](https://jseluis.github.io/silva-networks/paper/references/#ref-52), [53](https://jseluis.github.io/silva-networks/paper/references/#ref-53), [58](https://jseluis.github.io/silva-networks/paper/references/#ref-58). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [ ]:
from dataclasses import asdict

import torch
from torch import nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVADiffusionEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    all_silva_reproduction_specs,
    audit_silva_reproduction_specs,
    silva_reproduction_spec,
    validate_silva_transition,
)

torch.manual_seed(27)
assert audit_silva_reproduction_specs() == ()
specs = all_silva_reproduction_specs()
assert len(specs) == 30
[(spec.family, spec.source_relation, spec.verification_level) for spec in specs]

## Inspect the Complete Contract

The record separates scientific and numerical responsibilities:

| Field | Question answered |
| --- | --- |
| `equation` | What state-preserving map is solved? |
| `source_relation` | Is this native SILVA or a cited mechanism adaptation? |
| `datasets` and `preprocessing` | What observations enter the experiment? |
| `metrics` | What must be measured besides solver residual? |
| `notebooks` and `tests` | What executable evidence exists locally? |
| `configurable_parts` | Which operators and scale axes may be changed? |
| `constructor_signature` | Which exact public arguments are accepted? |

The residual

$$
r(z^\star,x)=T_\theta(z^\star,x)-z^\star
$$

checks the equilibrium equation. It does not measure classification accuracy,
field error, physical residual, FID, endpoint error, or reconstruction quality.

In [ ]:
for family in ("fno_deq", "mignn", "pideq", "deq_ddim"):
    spec = silva_reproduction_spec(family)
    print("\n", spec.family)
    print(" equation:", spec.equation)
    print(" data:", spec.datasets)
    print(" metrics:", spec.metrics)
    print(" signature:", spec.constructor_signature)

## Build a New Transition From Its Equation

For the compact transition

$$
T_\theta(z,x)=\tanh\!\left(W_xx+0.15\,h_\theta(z)\right),
$$

the source projection and recurrent field remain independently replaceable.
The validator checks shape, device, dtype, finiteness, and state-gradient
compatibility before the solver is introduced.

In [ ]:
class CustomTransition(nn.Module):
    def __init__(self, input_dim=2, state_dim=4):
        super().__init__()
        self.source = nn.Linear(input_dim, state_dim)
        self.recurrent = nn.Sequential(
            nn.Linear(state_dim, 8),
            nn.Tanh(),
            nn.Linear(8, state_dim),
        )

    def forward(self, state, inputs):
        return torch.tanh(self.source(inputs) + 0.15 * self.recurrent(state))


inputs = torch.linspace(-1.0, 1.0, 12).reshape(6, 2)
transition = CustomTransition()
report = validate_silva_transition(transition, torch.zeros(6, 4), inputs)
assert report.valid

custom = SILVAConditionedEquilibrium(
    transition,
    SILVAZeroInitializer(4),
    readout=nn.Linear(4, 1),
    config=SolverConfig(
        solver="anderson",
        max_iter=30,
        tol=1e-6,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
custom_result = custom(inputs, return_result=True)
custom_result.output.square().mean().backward()
assert custom_result.output.shape == (6, 1)
assert all(parameter.grad is not None for parameter in custom.parameters())
print("custom residual:", custom_result.solver_result.residual)

## Joint Diffusion Restoration Inside SILVA

Let $X=(x_{t_0},\ldots,x_{t_K})$ be one joint trajectory. A restoration
adaptation uses

$$
x_{t_{k+1}}^+
=P_{y,t_{k+1}}\!\left(D_{t_k\rightarrow t_{k+1}}
(x_{t_k};c,\xi_k)\right),
$$

where $D$ is a complete reverse step and $P$ is a declared measurement or
data-consistency operator. The triangular transition updates all trajectory
positions from the previous solver state. The step, observation operator,
schedule, stochastic terms, condition, and initial trajectory are separately
controllable.

In [ ]:
class CompleteReverseStep(nn.Module):
    def __init__(self):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(0.4))

    def forward(self, state, timestep, next_timestep, condition, noise):
        del timestep, next_timestep, noise
        return self.scale * state + condition


class ObservationOperator(nn.Module):
    def __init__(self):
        super().__init__()
        self.logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, candidate, observation, next_timestep):
        del next_timestep
        weight = self.logit.sigmoid()
        return weight * candidate + (1.0 - weight) * observation


restoration = SILVADiffusionEquilibrium(
    denoiser=None,
    alphas_cumprod=torch.tensor([0.95, 0.80, 0.60]),
    timesteps=(2, 1, 0),
    step_operator=CompleteReverseStep(),
    data_consistency=ObservationOperator(),
    config=SolverConfig(
        solver="picard",
        max_iter=5,
        tol=1e-7,
        backward_mode="unrolled",
        anderson_batch_dims=0,
    ),
)
noise = torch.randn(2, 1, 4, 4, requires_grad=True)
condition = torch.full_like(noise, 0.1, requires_grad=True)
observation = torch.zeros_like(noise, requires_grad=True)
restoration_result = restoration(
    noise,
    condition=condition,
    observation=observation,
    return_result=True,
)
restoration_result.output.square().mean().backward()
assert restoration_result.trajectory.shape == (3, *noise.shape)
assert observation.grad is not None
print("restoration residual:", restoration_result.solver_result.residual)

## Record What Was Actually Run

A benchmark record must preserve the source relationship and every deviation
from the cited protocol. This prevents a compact mechanism check from being
mistaken for a published-scale result and makes controlled extensions possible.

In [ ]:
selected = silva_reproduction_spec("deq_ddim")
run_record = {
    "family": selected.family,
    "paper_refs": selected.paper_refs,
    "source_relation": selected.source_relation,
    "verification_level": selected.verification_level,
    "dataset": "deterministic compact tensors",
    "split": "single checked batch",
    "model_options": {
        "trajectory_steps": 3,
        "complete_step": "CompleteReverseStep",
        "observation_operator": "ObservationOperator",
    },
    "solver": asdict(restoration.config),
    "metrics": {
        "fixed_point_residual": restoration_result.solver_result.residual,
        "output_norm": float(restoration_result.output.detach().norm()),
    },
    "seed": 27,
    "deviations": "compact mechanism check; no published image benchmark claimed",
}
assert run_record["metrics"]["fixed_point_residual"] < 1e-5
run_record

## From 27 Reproducing Silva And Source Methods to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the tensor solved to equilibrium |
| Condition | the observed input or source tensor |
| Repeated computation | the state-preserving transition evaluated by the root solver |
| Required invariants | shape, device, dtype, finiteness, and differentiability |
| Replaceable components | initializer, source encoder, transition, readout, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**fixed-point residual and task error against a deterministic target**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **state width, batch size, and data volume**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '27_reproducing_silva_and_source_methods.ipynb',
    "state": 'the tensor solved to equilibrium',
    "condition": 'the observed input or source tensor',
    "transition": 'the state-preserving transition evaluated by the root solver',
    "invariants": 'shape, device, dtype, finiteness, and differentiability',
    "compact_metric": 'fixed-point residual and task error against a deterministic target',
    "scale_axis": 'state width, batch size, and data volume',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


## Where to Go Next

| Question | Page |
| --- | --- |
| How are source methods represented without losing SILVA structure? | [Reproducing SILVA and Source Methods](https://jseluis.github.io/silva-networks/learn/reproducing-silva-and-papers/) |
| Which source-aware records and builders are public? | [Reproducibility API](https://jseluis.github.io/silva-networks/api/reproducibility/) |
| How should the resulting experiment be scaled? | [Full-Scale SILVA](https://jseluis.github.io/silva-networks/learn/full-scale-silva/) |
